In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import re

# Read all parquet files
files = glob.glob("../data/job_postings/*.parquet")
if not files:
    # Try S3 path via boto3
    import boto3
    import os
    from dotenv import load_dotenv
    load_dotenv("../.env")
    print("No local files found. Reading from S3...")
else:
    df = pd.concat([pd.read_parquet(f) for f in files])
    print(f"Total jobs loaded: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    df.head()

Total jobs loaded: 34


,job_id,title,company,location,state,country,is_remote,salary_min,salary_max,salary_currency,posted_at,required_skills,description_snippet,work_type
0,uoPXcsirkDGbT3SwAAAAAA==,"Data Engineering Manager III, AWS Supply Chain...",Amazon.com Services LLC,Arlington,Virginia,US,False,170000.0,230000.0,NaN,2026-03-29T07:00:00.000Z,NaN,DESCRIPTION\n\nWe are looking for a seasoned D...,On-site / Hybrid
1,M13678ldLC1s8v-uAAAAAA==,Data Engineer - NAVSUP OIS - Onsite,GDIT,Yorktown,Virginia,US,False,NaN,NaN,NaN,2026-03-29T16:00:00.000Z,NaN,Type of Requisition:\nRegular\n\nClearance Lev...,On-site / Hybrid
2,ZmW7hHn7floHrmzNAAAAAA==,Principal Data Engineer - Full-time,Leidos,Bethesda,Maryland,US,False,NaN,NaN,NaN,2026-03-29T01:00:00.000Z,NaN,**Description**\n• *Leidos** has an exciting o...,On-site / Hybrid
3,wD3izEtoYdmVeDGmAAAAAA==,"Data Engineer / ADF Administrator (Remote, Imm...",Upwork,NaN,NaN,US,True,NaN,NaN,NaN,2026-03-29T03:00:00.000Z,NaN,Overview\n\nWe are seeking an experienced Data...,Remote
4,YqgbwBqg9JPHPQZtAAAAAA==,ML Data Engineer,Openkyber,NaN,Texas,US,False,NaN,NaN,NaN,2026-03-29T04:00:00.000Z,NaN,"Hi, Greetings from OpenKyber! We reaching out ...",On-site / Hybrid


In [ ]:
plt.figure(figsize=(10,5))
top_companies = df['company'].value_counts().head(10)
top_companies.plot(kind='barh', color='steelblue')
plt.title('Top 10 Hiring Companies', fontsize=14)
plt.xlabel('Number of Postings')
plt.tight_layout()
plt.savefig('../docs/top_companies.png', dpi=150)
plt.show()
print(top_companies)

=== Top Hiring Companies ===
company
Amazon.com Services LLC      2
aPriori Technologies Inc     2
Lensa                        2
Amazon Web Services, Inc.    2
GDIT                         1
Leidos                       1
Upwork                       1
Openkyber                    1
Deloitte                     1
Dataeconomy                  1
Name: count, dtype: int64


In [ ]:
plt.figure(figsize=(6,6))
work_type = df['work_type'].value_counts()
colors = ['#2ec4b6', '#e76f51']
plt.pie(work_type.values, labels=work_type.index, autopct='%1.1f%%', colors=colors)
plt.title('Remote vs On-site / Hybrid', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/work_type.png', dpi=150)
plt.show()

=== Work Type Distribution ===
work_type
On-site / Hybrid    31
Remote               3
Name: count, dtype: int64


In [ ]:
# Jobs by state
plt.figure(figsize=(10,5))
top_states = df['state'].value_counts().dropna().head(10)
top_states.plot(kind='bar', color='#6c63ff')
plt.title('Top 10 States Hiring Data Professionals', fontsize=14)
plt.xlabel('State')
plt.ylabel('Job Postings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../docs/top_states.png', dpi=150)
plt.show()

=== Top States Hiring ===
state
Virginia                7
District of Columbia    5
Maryland                3
Massachusetts           3
New Jersey              3
Connecticut             2
Illinois                2
Texas                   1
Florida                 1
Pennsylvania            1
Name: count, dtype: int64


In [ ]:
# Salary insights (where available)
salary_df = df[df['salary_min'].notna() & df['salary_max'].notna()].copy()
salary_df['salary_avg'] = (salary_df['salary_min'] + salary_df['salary_max']) / 2

print(f"Jobs with salary data: {len(salary_df)} / {len(df)}")
print(f"Avg min salary: ${salary_df['salary_min'].mean():,.0f}")
print(f"Avg max salary: ${salary_df['salary_max'].mean():,.0f}")
print(f"Highest paying role: {salary_df.loc[salary_df['salary_avg'].idxmax(), 'title']}")
print(f"  at {salary_df.loc[salary_df['salary_avg'].idxmax(), 'company']}")
print(f"  salary: ${salary_df['salary_avg'].max():,.0f}")

if len(salary_df) > 0:
    plt.figure(figsize=(10,5))
    salary_df.sort_values('salary_avg', ascending=True).tail(10).plot(
        kind='barh', x='title', y='salary_avg', color='#f4a261', legend=False)
    plt.title('Top 10 Roles by Avg Salary', fontsize=14)
    plt.xlabel('Average Salary ($)')
    plt.tight_layout()
    plt.savefig('../docs/salary.png', dpi=150)
    plt.show()

Jobs with salary data: 5
Avg min salary: $131,400
Avg max salary: $182,800


In [ ]:
# Extract skills from description snippets
skill_keywords = [
    'python', 'sql', 'spark', 'kafka', 'aws', 'azure', 'gcp', 
    'airflow', 'dbt', 'snowflake', 'tableau', 'power bi', 
    'kubernetes', 'docker', 'terraform', 'hadoop', 'scala',
    'databricks', 'redshift', 'bigquery', 'pyspark'
]

skill_counts = {}
for skill in skill_keywords:
    count = df['description_snippet'].str.lower().str.contains(
        skill, na=False).sum()
    if count > 0:
        skill_counts[skill] = count

skill_series = pd.Series(skill_counts).sort_values(ascending=False)

plt.figure(figsize=(12,5))
skill_series.plot(kind='bar', color='#2ec4b6')
plt.title('Most In-Demand Skills (from Job Descriptions)', fontsize=14)
plt.xlabel('Skill')
plt.ylabel('Mentions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../docs/skills.png', dpi=150)
plt.show()
print("\nTop skills found:")
print(skill_series)

In [ ]:
print("=" * 50)
print("JOB MARKET PIPELINE — ANALYSIS SUMMARY")
print("=" * 50)
print(f"Total jobs analyzed:     {len(df)}")
print(f"Unique companies:        {df['company'].nunique()}")
print(f"Unique states:           {df['state'].nunique()}")
print(f"Remote jobs:             {(df['work_type'] == 'Remote').sum()}")
print(f"On-site/Hybrid jobs:     {(df['work_type'] == 'On-site / Hybrid').sum()}")
print(f"Jobs with salary data:   {df['salary_min'].notna().sum()}")
if df['salary_min'].notna().sum() > 0:
    print(f"Avg salary range:        ${df['salary_min'].mean():,.0f} - ${df['salary_max'].mean():,.0f}")
print(f"\nTop hiring company:      {df['company'].value_counts().index[0]}")
print(f"Top state:               {df['state'].value_counts().index[0]}")
print(f"\nTop in-demand skill:     {skill_series.index[0].upper()}")
print("=" * 50)